# Bayesian Hyperparameter Tuning for Novel Uncertainty-Aware Stacking Models

This notebook performs Bayesian hyperparameter tuning for 7 uncertainty-aware stacking configurations. The tuning includes base learners, entropy-based uncertainty meta-features, and the Logistic Regression meta-learner.


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

# Reproducibility
RANDOM_STATE = 42
N_SPLITS = 5

# Before running this notebook:
# 1) Place the cleaned and outlier-processed dataset in the Data folder.
# 2) Update DATA_PATH if your local file name or location is different.
DATA_PATH = Path("../Data/Partition class 0 Five_5 cluster on cleaned_data_afterDuplicate_tranform_dummies_numerical_STD_drop_Outlier.csv")
TARGET_COLUMN = "HeartDisease"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Please place the cleaned dataset in the Data folder or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("Dataset shape:", df.shape)
print("Feature matrix:", X.shape)
print("Target distribution:")
print(y.value_counts())


# Tuning hyperparameter tuning using Bayesian optimization LR+ET Novel Stacking

In [ ]:


import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.utils.multiclass import type_of_target
from imblearn.combine import SMOTEENN

try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as e:
    raise ImportError("Optuna must be installed first: pip install optuna") from e


# ---------- Uncertainty ----------
def entropy_binary(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p*np.log(p) + (1-p)*np.log(1-p))


# ---------- Helper: auto inner CV splits ----------
def _auto_inner_splits(y, want=3):
    y_arr = np.asarray(y).ravel()
    assert type_of_target(y_arr) in ("binary", "multiclass")
    _, cnt = np.unique(y_arr, return_counts=True)
    return max(2, min(int(cnt.min()), int(want)))


# ---------- Combo score: 0.5*Recall + 0.5*ROC-AUC ----------
def _combo_score(pipe, X, y, cv):
    rec = cross_val_score(pipe, X, y, cv=cv, scoring="recall",
                          n_jobs=-1, error_score=np.nan)
    auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc",
                          n_jobs=-1, error_score=np.nan)
    mrec = np.nanmean(rec)
    mauc = np.nanmean(auc)
    return 0.5 * (mrec if np.isfinite(mrec) else -np.inf) + \
           0.5 * (mauc if np.isfinite(mauc) else -np.inf)


# ---------- Bayesian tuning: LR (liblinear L1/L2) ----------
def tune_lr_bayes(X_train_res, y_train_res, n_trials=30, cv_splits=3, random_state=RANDOM_STATE):
    inner_k = _auto_inner_splits(y_train_res, want=cv_splits)
    inner_cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=random_state)
    sampler = TPESampler(seed=random_state)

    def objective(trial):
        penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
        C = trial.suggest_float("C", 1e-3, 1e+2, log=True)
        # After applying SMOTEENN, class_weight='balanced' may over-correct; None is preferred.
        class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
        clf = LogisticRegression(
            solver="liblinear", penalty=penalty, C=C,
            class_weight=class_weight, random_state=RANDOM_STATE, max_iter=1000
        )
        pipe = make_pipeline(StandardScaler(), clf)
        return _combo_score(pipe, X_train_res, y_train_res, inner_cv)

    study = optuna.create_study(direction="maximize", study_name="LR_Bayes", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    best_clf = LogisticRegression(
        solver="liblinear", penalty=best["penalty"], C=best["C"],
        class_weight=best["class_weight"], random_state=RANDOM_STATE, max_iter=1000
    )
    return best, study  # Return parameters to be used for OOF fitting.


# ---------- Bayesian tuning: ET ----------
def tune_et_bayes(X_train_res, y_train_res, n_trials=30, cv_splits=3, random_state=RANDOM_STATE):
    inner_k = _auto_inner_splits(y_train_res, want=cv_splits)
    inner_cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=random_state)
    sampler = TPESampler(seed=random_state)

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 50, 300),
            "max_depth":         trial.suggest_int("max_depth", 8, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "random_state":      0,
            "n_jobs":            -1
        }
        pipe = make_pipeline(StandardScaler(), ExtraTreesClassifier(**params))
        return _combo_score(pipe, X_train_res, y_train_res, inner_cv)

    study = optuna.create_study(direction="maximize", study_name="ET_Bayes", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best = study.best_params
    best.update({"random_state": 0, "n_jobs": -1})
    return best, study  # Return parameters to be used for OOF fitting.


# ---------- Create OOF meta-features on the resampled training set. ----------
def build_oof_meta_features(X_train_res, y_train_res, inner_splits, lr_best_params, et_best_params, seed=RANDOM_STATE):
    """
    Create OOF meta-features: 
      - Fit LR/ET using tuned parameters on the inner-training set.
      - Use predict_proba on the inner-validation set to obtain p_lr and p_et.
      - Compute entropy for each learner: H_lr and H_et.
      - Combine them into X_meta_tr (n_samples, 4) and y_meta_tr.
    """
    Xr = X_train_res
    yr = np.asarray(y_train_res).astype(int).ravel()
    k = _auto_inner_splits(yr, want=inner_splits)
    skf_in = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)

    n = Xr.shape[0]
    oof = np.zeros((n, 4), dtype=float)
    filled = np.zeros(n, dtype=bool)

    for tr, va in skf_in.split(Xr, yr):
        X_tr, X_va = Xr[tr], Xr[va]
        y_tr, y_va = yr[tr], yr[va]

        lr_clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                solver="liblinear",
                penalty=lr_best_params["penalty"],
                C=lr_best_params["C"],
                class_weight=lr_best_params["class_weight"],
                random_state=RANDOM_STATE, max_iter=1000
            )
        ).fit(X_tr, y_tr)

        et_clf = make_pipeline(
            StandardScaler(),
            ExtraTreesClassifier(**et_best_params)
        ).fit(X_tr, y_tr)

        p_lr  = lr_clf.predict_proba(X_va)[:, 1]
        p_et  = et_clf.predict_proba(X_va)[:, 1]
        H_lr  = entropy_binary(p_lr)
        H_et  = entropy_binary(p_et)

        oof[va, 0] = p_lr
        oof[va, 1] = H_lr
        oof[va, 2] = p_et
        oof[va, 3] = H_et
        filled[va] = True

    assert filled.all(), "OOF meta-features are incomplete for some samples."
    return oof, yr


# ---------- Select the meta-learner threshold to maximize recall; if tied, choose higher precision. ----------
def select_threshold_max_recall(y_true, proba, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 91)  # step size 0.01
    best_t, best_rec, best_prec = 0.5, -1.0, -1.0
    for t in grid:
        pred = (proba >= t).astype(int)
        rec = recall_score(y_true, pred, zero_division=0)
        prec = precision_score(y_true, pred, zero_division=0)
        if (rec > best_rec) or (np.isclose(rec, best_rec) and prec > best_prec):
            best_t, best_rec, best_prec = t, rec, prec
    return best_t, best_rec, best_prec


# ---------- Outer CV + Novel Stacking ----------
def novel_stacking_lr_et_with_bayes_oof_thresh(
        X, y, outer_splits=5, random_state=42,
        bayes_trials=25, bayes_cv=3, tuner_seed=RANDOM_STATE, inner_splits_oof=3
    ):
    X_arr = X.values if hasattr(X, "iloc") else np.asarray(X)
    y_arr = y.values if hasattr(y, "iloc") else np.asarray(y).astype(int).ravel()

    skf = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    fold_metrics, fold_confmats = [], []

    print("===== 5-Fold Novel Stacking (LR+ET prob+entropy -> LR meta) + SMOTEENN (train only) =====")
    print("===== Bayesian tuning for LR & ET (objective: 0.5*Recall + 0.5*ROC-AUC) =====")
    print("===== OOF meta-features + Meta threshold chosen on meta-train to maximize recall =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_arr, y_arr), start=1):
        X_tr, X_te = X_arr[tr_idx], X_arr[te_idx]
        y_tr, y_te = y_arr[tr_idx], y_arr[te_idx]

        # Apply SMOTEENN only to the training set.
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ---- Tune LR and ET on the resampled training set. ----
        lr_best_params, lr_study = tune_lr_bayes(
            X_tr_res, y_tr_res, n_trials=bayes_trials, cv_splits=bayes_cv, random_state=tuner_seed
        )
        et_best_params, et_study = tune_et_bayes(
            X_tr_res, y_tr_res, n_trials=bayes_trials, cv_splits=bayes_cv, random_state=tuner_seed
        )

        # ---- OOF meta-features on the resampled training set. ----
        X_meta_tr, y_meta_tr = build_oof_meta_features(
            X_tr_res, y_tr_res, inner_splits=inner_splits_oof,
            lr_best_params=lr_best_params, et_best_params=et_best_params, seed=tuner_seed
        )

        # ---- Meta-learner (LR fixed) ----
        meta_lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
        meta_lr.fit(X_meta_tr, y_meta_tr)

        # ---- Select threshold from meta-training data to maximize recall. ----
        proba_tr_meta = meta_lr.predict_proba(X_meta_tr)[:, 1]
        best_t, best_rec_tr, best_prec_tr = select_threshold_max_recall(y_meta_tr, proba_tr_meta)

        # ---- Fit base learners on the full resampled training set using the best parameters. ----
        lr_full = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                solver="liblinear",
                penalty=lr_best_params["penalty"],
                C=lr_best_params["C"],
                class_weight=lr_best_params["class_weight"],
                random_state=RANDOM_STATE, max_iter=1000
            )
        ).fit(X_tr_res, y_tr_res)

        et_full = make_pipeline(
            StandardScaler(),
            ExtraTreesClassifier(**et_best_params)
        ).fit(X_tr_res, y_tr_res)

        # ---- Create meta-features on the original test fold. ----
        p_lr_te = lr_full.predict_proba(X_te)[:, 1]; H_lr_te = entropy_binary(p_lr_te)
        p_et_te = et_full.predict_proba(X_te)[:, 1]; H_et_te = entropy_binary(p_et_te)
        X_meta_te = np.column_stack([p_lr_te, H_lr_te, p_et_te, H_et_te])

        # ---- Predict using the meta-learner and the selected threshold. ----
        y_proba = meta_lr.predict_proba(X_meta_te)[:, 1]
        y_pred  = (y_proba >= best_t).astype(int)

        # ---- Metrics ----
        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec  = recall_score(y_te, y_pred, zero_division=0)
        f1   = f1_score(y_te, y_pred, zero_division=0)
        auc  = roc_auc_score(y_te, y_proba)
        cm   = confusion_matrix(y_te, y_pred)

        fold_confmats.append(cm)
        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc,
            "meta_thresh": float(best_t),
            "TN": int(cm[0,0]), "FP": int(cm[0,1]), "FN": int(cm[1,0]), "TP": int(cm[1,1]),
            "lr_best": str(lr_study.best_params),
            "et_best": str(et_study.best_params),
        })

        print(f"[Fold {fold_idx}] Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}  | meta_th={best_t:.2f} (train recall={best_rec_tr:.4f}, prec={best_prec_tr:.4f})")
        print(f"  Confusion Matrix:\n{cm}")
        print(f"  LR best params: {lr_study.best_params}")
        print(f"  ET best params: {et_study.best_params}\n")

    # ---- Summary ----
    df_res = pd.DataFrame(fold_metrics)
    print("===== Summary over 5 folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        print(f"{m.capitalize():<9}: mean={df_res[m].mean():.4f}  std={df_res[m].std(ddof=1):.4f}")

    print("\nPer-fold threshold / TN/FP/FN/TP:")
    print(df_res[["fold","meta_thresh","TN","FP","FN","TP"]].to_string(index=False))

    print("\nBest params per fold:")
    print(df_res[["fold","lr_best","et_best"]].to_string(index=False))

    return df_res


# ---------- Run the function and display results. ----------
# Note: X and y must already be prepared (DataFrame/Series or NumPy array).

df_results = novel_stacking_lr_et_with_bayes_oof_thresh(
    X, y,
    outer_splits=5,
    random_state=42,
    bayes_trials=25,   # Increase if computational time is available.
    bayes_cv=3,
    tuner_seed=RANDOM_STATE,
    inner_splits_oof=3
)

# Display overall results.
print("\n=== Metrics per fold ===")
print(df_results[["fold","accuracy","precision","recall","f1","auc","meta_thresh"]]
      .round(4).to_string(index=False))

print("\n=== Best params per fold ===")
print(df_results[["fold","lr_best","et_best"]].to_string(index=False))


# Tuning hyperparameter tuning using Bayesian optimization SVM+ET Novel Stacking

In [ ]:

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.utils.multiclass import type_of_target

from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as e:
    raise ImportError("Optuna must be installed first: pip install optuna") from e


# ---------- Uncertainty ----------
def entropy_binary(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p*np.log(p) + (1-p)*np.log(1-p))


# ---------- Helpers ----------
def _auto_inner_splits(y, want=3):
    y_arr = np.asarray(y).ravel()
    assert type_of_target(y_arr) in ("binary", "multiclass")
    _, cnt = np.unique(y_arr, return_counts=True)
    return max(2, min(int(cnt.min()), int(want)))

def _combo_score(pipe, X, y, cv):
    rec = cross_val_score(pipe, X, y, cv=cv, scoring="recall",
                          n_jobs=-1, error_score=np.nan)
    auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc",
                          n_jobs=-1, error_score=np.nan)
    mrec = np.nanmean(rec); mauc = np.nanmean(auc)
    mrec = -np.inf if not np.isfinite(mrec) else mrec
    mauc = -np.inf if not np.isfinite(mauc) else mauc
    return 0.5*mrec + 0.5*mauc

def select_threshold_max_recall(y_true, proba, grid=None):
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    best_t, best_rec, best_prec = 0.5, -1.0, -1.0
    for t in grid:
        pred = (proba >= t).astype(int)
        rec = recall_score(y_true, pred, zero_division=0)
        prec = precision_score(y_true, pred, zero_division=0)
        if (rec > best_rec) or (np.isclose(rec, best_rec) and prec > best_prec):
            best_t, best_rec, best_prec = t, rec, prec
    return float(best_t), float(best_rec), float(best_prec)


# ---------- Bayesian tuning on the raw outer-training set with inner SMOTEENN. ----------
def tune_svm_bayes(X_train_raw, y_train_raw, n_trials=30, cv_splits=3, random_state=RANDOM_STATE):
    inner_k  = _auto_inner_splits(y_train_raw, want=cv_splits)
    inner_cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=random_state)
    sampler  = TPESampler(seed=random_state)

    def objective(trial):
        C = trial.suggest_float("C", 1e-3, 1e+2, log=True)
        class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
        pipe = ImbPipeline([
            ("smoteenn", SMOTEENN(random_state=random_state)),
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE,
                        C=C, class_weight=class_weight))
        ])
        return _combo_score(pipe, X_train_raw, y_train_raw, inner_cv)

    study = optuna.create_study(direction="maximize", study_name="SVM_Bayes", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study

def tune_et_bayes(X_train_raw, y_train_raw, n_trials=30, cv_splits=3, random_state=RANDOM_STATE):
    inner_k  = _auto_inner_splits(y_train_raw, want=cv_splits)
    inner_cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=random_state)
    sampler  = TPESampler(seed=random_state)

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 50, 300),
            "max_depth":         trial.suggest_int("max_depth", 8, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "random_state":      0,
            "n_jobs":            -1
        }
        pipe = ImbPipeline([
            ("smoteenn", SMOTEENN(random_state=random_state)),
            ("scaler", StandardScaler()),
            ("et", ExtraTreesClassifier(**params))
        ])
        return _combo_score(pipe, X_train_raw, y_train_raw, inner_cv)

    study = optuna.create_study(direction="maximize", study_name="ET_Bayes", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study


# ---------- OOF meta-features on the resampled outer-training set. ----------
def build_oof_meta_features_svm_et(X_train_res, y_train_res,
                                   svm_best, et_best,
                                   inner_splits=3, seed=RANDOM_STATE):
    Xr = X_train_res
    yr = np.asarray(y_train_res).astype(int).ravel()
    k = _auto_inner_splits(yr, want=inner_splits)
    skf_in = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)

    n = Xr.shape[0]
    # [p_svm, H_svm, p_et, H_et]
    oof = np.zeros((n, 4), dtype=float)
    filled = np.zeros(n, dtype=bool)

    for tr, va in skf_in.split(Xr, yr):
        X_tr, X_va = Xr[tr], Xr[va]
        y_tr, y_va = yr[tr], yr[va]

        svm = ImbPipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE,
                        C=svm_best.get("C", 1.0),
                        class_weight=svm_best.get("class_weight", None)))
        ])
        et  = ImbPipeline([
            ("scaler", StandardScaler()),
            ("et", ExtraTreesClassifier(**{**et_best, "random_state":0, "n_jobs":-1}))
        ])

        svm.fit(X_tr, y_tr); et.fit(X_tr, y_tr)

        p_svm = svm.predict_proba(X_va)[:, 1]; H_svm = entropy_binary(p_svm)
        p_et  = et.predict_proba (X_va)[:, 1]; H_et  = entropy_binary(p_et)

        oof[va, 0] = p_svm; oof[va, 1] = H_svm
        oof[va, 2] = p_et;  oof[va, 3] = H_et
        filled[va] = True

    assert filled.all(), "OOF meta-features are incomplete for some samples."
    return oof, yr


# ---------- Outer CV ----------
def novel_stacking_svm_et_with_bayes_oof_thresh(
        X, y, outer_splits=5, random_state=42,
        bayes_trials=25, bayes_cv=3, tuner_seed=RANDOM_STATE, inner_splits_oof=3
    ):
    X_arr = X.values if hasattr(X, "iloc") else np.asarray(X)
    y_arr = y.values if hasattr(y, "iloc") else np.asarray(y).astype(int).ravel()

    skf = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    fold_metrics, fold_confmats = [], []

    print("===== 5-Fold Novel Stacking (SVM entropy + ET entropy -> LR meta) + SMOTEENN (outer-train only) =====")
    print("===== Bayesian tuning objective = 0.5*Recall + 0.5*AUC on RAW data (inner CV has SMOTEENN) =====")
    print("===== OOF meta-features + Meta threshold (maximize recall on meta-train) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_arr, y_arr), start=1):
        X_tr_raw, X_te = X_arr[tr_idx], X_arr[te_idx]
        y_tr_raw, y_te = y_arr[tr_idx], y_arr[te_idx]

        # 1) Resample only the outer-training set.
        smenn = SMOTEENN(random_state=RANDOM_STATE)
        X_tr_res, y_tr_res = smenn.fit_resample(X_tr_raw, y_tr_raw)

        # 2) Tune SVM/ET on the raw outer-training set; inner CV includes SMOTEENN.
        svm_best, svm_study = tune_svm_bayes(X_tr_raw, y_tr_raw, bayes_trials, bayes_cv, tuner_seed)
        et_best,  et_study  = tune_et_bayes (X_tr_raw, y_tr_raw, bayes_trials, bayes_cv, tuner_seed)

        X_meta_tr, y_meta_tr = build_oof_meta_features_svm_et(
            X_tr_res, y_tr_res, svm_best, et_best,
            inner_splits=inner_splits_oof, seed=tuner_seed
        )

        # 4) Train the meta-learner and select the threshold on meta-training data.
        meta_lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
        meta_lr.fit(X_meta_tr, y_meta_tr)
        proba_tr_meta = meta_lr.predict_proba(X_meta_tr)[:, 1]
        best_t, best_rec_tr, best_prec_tr = select_threshold_max_recall(y_meta_tr, proba_tr_meta)

        # 5) Fit base learners on the full resampled training set and create meta-test features.
        svm_full = ImbPipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE,
                        C=svm_best.get("C", 1.0),
                        class_weight=svm_best.get("class_weight", None)))
        ])
        et_full  = ImbPipeline([
            ("scaler", StandardScaler()),
            ("et", ExtraTreesClassifier(**{**et_best, "random_state":0, "n_jobs":-1}))
        ])

        svm_full.fit(X_tr_res, y_tr_res)
        et_full.fit (X_tr_res, y_tr_res)

        p_svm_te = svm_full.predict_proba(X_te)[:, 1]; H_svm_te = entropy_binary(p_svm_te)
        p_et_te  = et_full.predict_proba (X_te)[:, 1]; H_et_te  = entropy_binary(p_et_te)
        X_meta_te = np.column_stack([p_svm_te, H_svm_te, p_et_te, H_et_te])

        # 6) Predict using the selected threshold.
        y_proba = meta_lr.predict_proba(X_meta_te)[:, 1]
        y_pred  = (y_proba >= best_t).astype(int)

        # 7) Metrics
        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec  = recall_score(y_te, y_pred, zero_division=0)
        f1   = f1_score(y_te, y_pred, zero_division=0)
        auc  = roc_auc_score(y_te, y_proba)
        cm   = confusion_matrix(y_te, y_pred)

        fold_confmats.append(cm)
        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc,
            "meta_thresh": best_t,
            "TN": int(cm[0,0]), "FP": int(cm[0,1]), "FN": int(cm[1,0]), "TP": int(cm[1,1]),
            "svm_best": str(svm_study.best_params),
            "et_best":  str(et_study.best_params),
        })

        print(f"[Fold {fold_idx}] Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f} | meta_th={best_t:.2f} (train recall={best_rec_tr:.4f}, prec={best_prec_tr:.4f})")
        print(f"  Confusion Matrix:\n{cm}")
        print(f"  SVM best params: {svm_study.best_params}")
        print(f"  ET  best params: {et_study.best_params}\n")

    # Summary
    df_res = pd.DataFrame(fold_metrics)
    print("===== Summary over 5 folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        print(f"{m.capitalize():<9}: mean={df_res[m].mean():.4f}  std={df_res[m].std(ddof=1):.4f}")

    print("\nPer-fold threshold / TN/FP/FN/TP:")
    print(df_res[["fold","meta_thresh","TN","FP","FN","TP"]].to_string(index=False))

    print("\nBest params per fold:")
    print(df_res[["fold","svm_best","et_best"]].to_string(index=False))

    return df_res


# ---------- Run the function and display results. ----------
# X and y must be prepared beforehand (DataFrame/Series or NumPy array).

df_results = novel_stacking_svm_et_with_bayes_oof_thresh(
    X, y,
    outer_splits=5,
    random_state=42,
    bayes_trials=25,   # Increase if time is available.
    bayes_cv=3,
    tuner_seed=RANDOM_STATE,
    inner_splits_oof=3
)

print("\n=== Metrics per fold ===")
print(df_results[["fold","accuracy","precision","recall","f1","auc","meta_thresh"]]
      .round(4).to_string(index=False))

print("\n=== Best params per fold ===")
print(df_results[["fold","svm_best","et_best"]].to_string(index=False))


# Tuning hyperparameter tuning using Bayesian optimization LR+SVM+ET Novel Stacking

In [ ]:

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
from sklearn.utils.multiclass import type_of_target

from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline  # Important: allows SMOTEENN inside CV training.

try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError as e:
    raise ImportError("Optuna must be installed first: pip install optuna") from e


# ---------- Uncertainty ----------
def entropy_binary(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p*np.log(p) + (1-p)*np.log(1-p))


# ---------- Helpers ----------
def _auto_inner_splits(y, want=3):
    y_arr = np.asarray(y).ravel()
    assert type_of_target(y_arr) in ("binary", "multiclass")
    _, cnt = np.unique(y_arr, return_counts=True)
    return max(2, min(int(cnt.min()), int(want)))

def _combo_score(pipe, X, y, cv):
    rec = cross_val_score(pipe, X, y, cv=cv, scoring="recall",
                          n_jobs=-1, error_score=np.nan)
    auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc",
                          n_jobs=-1, error_score=np.nan)
    mrec = np.nanmean(rec); mauc = np.nanmean(auc)
    mrec = -np.inf if not np.isfinite(mrec) else mrec
    mauc = -np.inf if not np.isfinite(mauc) else mauc
    return 0.5*mrec + 0.5*mauc

def select_threshold_max_recall(y_true, proba, grid=None):
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    best_t, best_rec, best_prec = 0.5, -1.0, -1.0
    for t in grid:
        pred = (proba >= t).astype(int)
        rec = recall_score(y_true, pred, zero_division=0)
        prec = precision_score(y_true, pred, zero_division=0)
        if (rec > best_rec) or (np.isclose(rec, best_rec) and prec > best_prec):
            best_t, best_rec, best_prec = t, rec, prec
    return float(best_t), float(best_rec), float(best_prec)


# ---------- Bayesian tuning on the raw outer-training set with inner SMOTEENN. ----------
def tune_svm_bayes(X_train_raw, y_train_raw, n_trials=30, cv_splits=3, random_state=RANDOM_STATE):
    inner_k  = _auto_inner_splits(y_train_raw, want=cv_splits)
    inner_cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=random_state)
    sampler  = TPESampler(seed=random_state)

    def objective(trial):
        C = trial.suggest_float("C", 1e-3, 1e+2, log=True)
        class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
        pipe = ImbPipeline([
            ("smoteenn", SMOTEENN(random_state=random_state)),
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE,
                        C=C, class_weight=class_weight))
        ])
        return _combo_score(pipe, X_train_raw, y_train_raw, inner_cv)

    study = optuna.create_study(direction="maximize", study_name="SVM_Bayes", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study

def tune_lr_bayes(X_train_raw, y_train_raw, n_trials=30, cv_splits=3, random_state=RANDOM_STATE):
    inner_k  = _auto_inner_splits(y_train_raw, want=cv_splits)
    inner_cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=random_state)
    sampler  = TPESampler(seed=random_state)

    def objective(trial):
        penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
        C = trial.suggest_float("C", 1e-3, 1e+2, log=True)
        class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
        pipe = ImbPipeline([
            ("smoteenn", SMOTEENN(random_state=random_state)),
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(solver="liblinear", penalty=penalty, C=C,
                                     class_weight=class_weight, random_state=RANDOM_STATE, max_iter=1000))
        ])
        return _combo_score(pipe, X_train_raw, y_train_raw, inner_cv)

    study = optuna.create_study(direction="maximize", study_name="LR_Bayes", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study

def tune_et_bayes(X_train_raw, y_train_raw, n_trials=30, cv_splits=3, random_state=RANDOM_STATE):
    inner_k  = _auto_inner_splits(y_train_raw, want=cv_splits)
    inner_cv = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=random_state)
    sampler  = TPESampler(seed=random_state)

    def objective(trial):
        params = {
            "n_estimators":      trial.suggest_int("n_estimators", 50, 300),
            "max_depth":         trial.suggest_int("max_depth", 8, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features":      trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "random_state":      0,
            "n_jobs":            -1
        }
        pipe = ImbPipeline([
            ("smoteenn", SMOTEENN(random_state=random_state)),
            ("scaler", StandardScaler()),
            ("et", ExtraTreesClassifier(**params))
        ])
        return _combo_score(pipe, X_train_raw, y_train_raw, inner_cv)

    study = optuna.create_study(direction="maximize", study_name="ET_Bayes", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params, study


# ---------- OOF meta-features on the resampled outer-training set. ----------
def build_oof_meta_features_svm_lr_et(X_train_res, y_train_res,
                                      svm_best, lr_best, et_best,
                                      inner_splits=3, seed=RANDOM_STATE):
    Xr = X_train_res
    yr = np.asarray(y_train_res).astype(int).ravel()
    k = _auto_inner_splits(yr, want=inner_splits)
    skf_in = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)

    n = Xr.shape[0]
    oof = np.zeros((n, 6), dtype=float)  # [p_svm, H_svm, p_lr, H_lr, p_et, H_et]
    filled = np.zeros(n, dtype=bool)

    for tr, va in skf_in.split(Xr, yr):
        X_tr, X_va = Xr[tr], Xr[va]
        y_tr, y_va = yr[tr], yr[va]

        svm = ImbPipeline([("scaler", StandardScaler()),
                           ("svm", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE,
                                       C=svm_best.get("C", 1.0),
                                       class_weight=svm_best.get("class_weight", None)))])
        lr  = ImbPipeline([("scaler", StandardScaler()),
                           ("lr", LogisticRegression(solver="liblinear",
                                                     penalty=lr_best.get("penalty","l2"),
                                                     C=lr_best.get("C",1.0),
                                                     class_weight=lr_best.get("class_weight", None),
                                                     random_state=RANDOM_STATE, max_iter=1000))])
        et  = ImbPipeline([("scaler", StandardScaler()),
                           ("et", ExtraTreesClassifier(**{**et_best, "random_state":0, "n_jobs":-1}))])

        svm.fit(X_tr, y_tr); lr.fit(X_tr, y_tr); et.fit(X_tr, y_tr)

        p_svm = svm.predict_proba(X_va)[:, 1]; H_svm = entropy_binary(p_svm)
        p_lr  = lr.predict_proba (X_va)[:, 1]; H_lr  = entropy_binary(p_lr)
        p_et  = et.predict_proba (X_va)[:, 1]; H_et  = entropy_binary(p_et)

        oof[va, 0] = p_svm; oof[va, 1] = H_svm
        oof[va, 2] = p_lr;  oof[va, 3] = H_lr
        oof[va, 4] = p_et;  oof[va, 5] = H_et
        filled[va] = True

    assert filled.all(), "OOF meta-features are incomplete for some samples."
    return oof, yr


# ---------- Outer CV ----------
def novel_stacking_svm_lr_et_with_bayes_oof_thresh(
        X, y, outer_splits=5, random_state=42,
        bayes_trials=25, bayes_cv=3, tuner_seed=RANDOM_STATE, inner_splits_oof=3
    ):
    X_arr = X.values if hasattr(X, "iloc") else np.asarray(X)
    y_arr = y.values if hasattr(y, "iloc") else np.asarray(y).astype(int).ravel()

    skf = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=random_state)
    fold_metrics, fold_confmats = [], []

    print("===== 5-Fold Novel Stacking (SVM/LR/ET prob+entropy -> LR meta) + SMOTEENN (outer-train only) =====")
    print("===== Bayesian tuning objective = 0.5*Recall + 0.5*AUC on RAW data w/ inner SMOTEENN =====")
    print("===== OOF meta-features + Select meta threshold on meta-train (maximize recall) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X_arr, y_arr), start=1):
        X_tr_raw, X_te = X_arr[tr_idx], X_arr[te_idx]
        y_tr_raw, y_te = y_arr[tr_idx], y_arr[te_idx]

        # 1) Resample only the outer-training set.
        smenn = SMOTEENN(random_state=RANDOM_STATE)
        X_tr_res, y_tr_res = smenn.fit_resample(X_tr_raw, y_tr_raw)

        # 2) Tune SVM/LR/ET on the raw outer-training set; inner CV includes SMOTEENN.
        svm_best, svm_study = tune_svm_bayes(X_tr_raw, y_tr_raw, bayes_trials, bayes_cv, tuner_seed)
        lr_best,  lr_study  = tune_lr_bayes (X_tr_raw, y_tr_raw, bayes_trials, bayes_cv, tuner_seed)
        et_best,  et_study  = tune_et_bayes (X_tr_raw, y_tr_raw, bayes_trials, bayes_cv, tuner_seed)

        X_meta_tr, y_meta_tr = build_oof_meta_features_svm_lr_et(
            X_tr_res, y_tr_res, svm_best, lr_best, et_best,
            inner_splits=inner_splits_oof, seed=tuner_seed
        )

        # 4) Train the meta-learner and select the threshold on meta-training data.
        meta_lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
        meta_lr.fit(X_meta_tr, y_meta_tr)
        proba_tr_meta = meta_lr.predict_proba(X_meta_tr)[:, 1]
        best_t, best_rec_tr, best_prec_tr = select_threshold_max_recall(y_meta_tr, proba_tr_meta)

        # 5) Fit base learners on the full resampled training set and create meta-test features.
        svm_full = ImbPipeline([("scaler", StandardScaler()),
                                ("svm", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE,
                                            C=svm_best.get("C", 1.0),
                                            class_weight=svm_best.get("class_weight", None)))])
        lr_full  = ImbPipeline([("scaler", StandardScaler()),
                                ("lr", LogisticRegression(solver="liblinear",
                                                          penalty=lr_best.get("penalty","l2"),
                                                          C=lr_best.get("C",1.0),
                                                          class_weight=lr_best.get("class_weight", None),
                                                          random_state=RANDOM_STATE, max_iter=1000))])
        et_full  = ImbPipeline([("scaler", StandardScaler()),
                                ("et", ExtraTreesClassifier(**{**et_best, "random_state":0, "n_jobs":-1}))])

        svm_full.fit(X_tr_res, y_tr_res)
        lr_full.fit (X_tr_res, y_tr_res)
        et_full.fit (X_tr_res, y_tr_res)

        p_svm_te = svm_full.predict_proba(X_te)[:, 1]; H_svm_te = entropy_binary(p_svm_te)
        p_lr_te  = lr_full.predict_proba (X_te)[:, 1]; H_lr_te  = entropy_binary(p_lr_te)
        p_et_te  = et_full.predict_proba (X_te)[:, 1]; H_et_te  = entropy_binary(p_et_te)

        X_meta_te = np.column_stack([p_svm_te, H_svm_te, p_lr_te, H_lr_te, p_et_te, H_et_te])

        # 6) Predict using the selected threshold.
        y_proba = meta_lr.predict_proba(X_meta_te)[:, 1]
        y_pred  = (y_proba >= best_t).astype(int)

        # 7) Metrics
        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, zero_division=0)
        rec  = recall_score(y_te, y_pred, zero_division=0)
        f1   = f1_score(y_te, y_pred, zero_division=0)
        auc  = roc_auc_score(y_te, y_proba)
        cm   = confusion_matrix(y_te, y_pred)

        fold_confmats.append(cm)
        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "auc": auc,
            "meta_thresh": best_t,
            "TN": int(cm[0,0]), "FP": int(cm[0,1]), "FN": int(cm[1,0]), "TP": int(cm[1,1]),
            "svm_best": str(svm_study.best_params),
            "lr_best":  str(lr_study.best_params),
            "et_best":  str(et_study.best_params),
        })

        print(f"[Fold {fold_idx}] Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f} | meta_th={best_t:.2f} (train recall={best_rec_tr:.4f}, prec={best_prec_tr:.4f})")
        print(f"  Confusion Matrix:\n{cm}")
        print(f"  SVM best params: {svm_study.best_params}")
        print(f"  LR  best params: {lr_study.best_params}")
        print(f"  ET  best params: {et_study.best_params}\n")

    # Summary
    df_res = pd.DataFrame(fold_metrics)
    print("===== Summary over 5 folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        print(f"{m.capitalize():<9}: mean={df_res[m].mean():.4f}  std={df_res[m].std(ddof=1):.4f}")

    print("\nPer-fold threshold / TN/FP/FN/TP:")
    print(df_res[["fold","meta_thresh","TN","FP","FN","TP"]].to_string(index=False))

    print("\nBest params per fold:")
    print(df_res[["fold","svm_best","lr_best","et_best"]].to_string(index=False))

    return df_res


# ---------- Run the function and display results. ----------
# X and y must already be prepared (DataFrame/Series or NumPy array).
df_results = novel_stacking_svm_lr_et_with_bayes_oof_thresh(
    X, y,
    outer_splits=5,
    random_state=42,
    bayes_trials=25,
    bayes_cv=3,
    tuner_seed=RANDOM_STATE,
    inner_splits_oof=3
)

print("\n=== Metrics per fold ===")
print(df_results[["fold","accuracy","precision","recall","f1","auc","meta_thresh"]]
      .round(4).to_string(index=False))

print("\n=== Best params per fold ===")
print(df_results[["fold","svm_best","lr_best","et_best"]].to_string(index=False))


# Tuning hyperparameter tuning using Bayesian optimization RF+LR+ET Novel Stacking

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
# 1) Utilities: entropy, RF vote variance, meta-feature builder
# =========================================================
def entropy_binary(p, eps=1e-12):
    """Binary probability entropy: H(p) = -(p log p + (1-p) log (1-p))"""
    p = np.clip(p, eps, 1 - eps)
    return -(p*np.log(p) + (1-p)*np.log(1-p))


def rf_vote_variance(rf_pipeline, X):
    """
    Estimate RF uncertainty using variance in votes across trees.
    - Use the scaler in the existing pipeline to transform X.
    - Extract hard votes (0/1) from each tree.
    - Compute p_hat as the proportion of votes equal to 1.
    - Return p_hat*(1-p_hat).
    """
    scaler = rf_pipeline.named_steps['standardscaler']
    X_trf = scaler.transform(X.values if hasattr(X, "iloc") else X)

    rf = rf_pipeline.named_steps['randomforestclassifier']
    votes = np.column_stack([est.predict(X_trf) for est in rf.estimators_])
    p_hat = votes.mean(axis=1)
    return p_hat * (1.0 - p_hat)


def build_base_models_from_params(params):
    """
    Build base models (LR, ET, RF) using tuned hyperparameters.
    Expected keys in params:
      lr_C, lr_max_iter
      et_n_estimators, et_max_depth, et_min_samples_split,
      et_min_samples_leaf, et_max_features, et_bootstrap
      rf_n_estimators, rf_max_depth, rf_min_samples_split,
      rf_min_samples_leaf, rf_max_features, rf_bootstrap
    """
    lr_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=params["lr_C"],
            max_iter=params["lr_max_iter"],
            random_state=RANDOM_STATE
        )
    )

    et_model = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params["et_n_estimators"],
            max_depth=params["et_max_depth"],
            min_samples_split=params["et_min_samples_split"],
            min_samples_leaf=params["et_min_samples_leaf"],
            max_features=params["et_max_features"],
            bootstrap=params["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    rf_model = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=params["rf_n_estimators"],
            max_depth=params["rf_max_depth"],
            min_samples_split=params["rf_min_samples_split"],
            min_samples_leaf=params["rf_min_samples_leaf"],
            max_features=params["rf_max_features"],
            bootstrap=params["rf_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    return lr_model, et_model, rf_model


def build_meta_from_params(params):
    """
    The meta-learner is LogisticRegression.
    Expected keys in params:
      meta_C, meta_max_iter
    """
    meta_lr = LogisticRegression(
        C=params["meta_C"],
        max_iter=params["meta_max_iter"],
        random_state=RANDOM_STATE
    )
    return meta_lr


def make_meta_features(lr_model, et_model, rf_model, X_input):
    """
    Create the meta-feature matrix for the meta-learner.
    Features:
      [ LR_prob, LR_entropy,
        ET_prob, ET_entropy,
        RF_prob, RF_voteVar ]
    """
    # LR
    p_lr = lr_model.predict_proba(X_input)[:, 1]
    H_lr = entropy_binary(p_lr)

    # ET
    p_et = et_model.predict_proba(X_input)[:, 1]
    H_et = entropy_binary(p_et)

    # RF
    p_rf = rf_model.predict_proba(X_input)[:, 1]
    Var_rf = rf_vote_variance(rf_model, X_input)

    X_meta = np.column_stack([
        p_lr,  H_lr,
        p_et,  H_et,
        p_rf,  Var_rf
    ])
    return X_meta


# =========================================================
# 2) Inner CV objective:
#    Evaluate params_joint using inner CV only within train_outer.
#    Objective: mean recall.
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
    Steps in each inner split:
      1. Split into inner_train and inner_val.
      2. Apply SMOTEENN to inner_train.
      3. Train base models (LR/ET/RF) using params_joint.
      4. Create meta-features from inner_train_resampled.
      5. Train meta LR using params_joint.
      6. Create meta-features for the raw inner_val set.
      7. Use meta LR to predict probabilities on inner_val and classify using the tuned threshold.
      8. Store recall.
    Return mean recall.
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== Train base models using params_joint. =====
        lr_base, et_base, rf_base = build_base_models_from_params(params_joint)
        lr_base.fit(X_tr_res, y_tr_res)
        et_base.fit(X_tr_res, y_tr_res)
        rf_base.fit(X_tr_res, y_tr_res)

        # ===== meta-train features (train_resampled) =====
        X_meta_tr = make_meta_features(lr_base, et_base, rf_base, X_tr_res)
        y_meta_tr = np.asarray(y_tr_res).astype(int).ravel()

        meta_lr = build_meta_from_params(params_joint)
        meta_lr.fit(X_meta_tr, y_meta_tr)

        # ===== Meta-validation features on the raw validation set without oversampling. =====
        X_meta_val = make_meta_features(lr_base, et_base, rf_base, X_val)

        y_val_proba = meta_lr.predict_proba(X_meta_val)[:, 1]

        thr = params_joint["meta_threshold"]
        y_val_pred = (y_val_proba >= thr).astype(int)

        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# 3) Bayesian tuning with Optuna for one outer fold.
#    -> Return best_params_joint and best_inner_recall.
# =========================================================
from sklearn.metrics import recall_score

def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
    Jointly tune the following components::
      - LR base
      - ET base
      - RF base
      - Meta LR
      - Threshold for meta LR.
    Objective: mean recall from inner CV.
    """

    def objective(trial):
        # -------- LR base params --------
        lr_C = trial.suggest_float("lr_C", 1e-4, 1e2, log=True)
        lr_max_iter = trial.suggest_int("lr_max_iter", 100, 2000)

        # -------- ET base params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- RF base params --------
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features = trial.suggest_categorical(
            "rf_max_features", ["sqrt", "log2", None]
        )
        rf_bootstrap = trial.suggest_categorical(
            "rf_bootstrap", [True, False]
        )

        # -------- meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        # -------- decision threshold --------
        meta_threshold = trial.suggest_float("meta_threshold", 0.5, 0.5)

        params_joint = {
            "lr_C": lr_C,
            "lr_max_iter": lr_max_iter,

            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            "rf_n_estimators": rf_n_estimators,
            "rf_max_depth": rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf": rf_min_samples_leaf,
            "rf_max_features": rf_max_features,
            "rf_bootstrap": rf_bootstrap,

            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter,

            "meta_threshold": meta_threshold
        }

        mean_rec = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        return mean_rec  # maximize recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()
    best_inner_recall = study.best_value

    return best_params_joint, best_inner_recall


# =========================================================
# 4) MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials: Number of Optuna trials per outer fold.
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV (joint Bayesian tuning for LR+ET+RF base -> LR meta with uncertainty features) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        # ==========================
        # split outer fold
        # ==========================
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # ==========================
        # 1) Tune on train_outer.
        # ==========================
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx  # Use a different seed for each fold.
        )

        fold_params.append(best_params_fold)

        # ==========================
        # 2) Retrain using the best parameters on the full train_outer set.
        # ==========================
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        # base models
        lr_best, et_best, rf_best = build_base_models_from_params(best_params_fold)
        lr_best.fit(X_train_res, y_train_res)
        et_best.fit(X_train_res, y_train_res)
        rf_best.fit(X_train_res, y_train_res)

        # meta-train features
        X_meta_train_res = make_meta_features(lr_best, et_best, rf_best, X_train_res)
        y_meta_train_res = np.asarray(y_train_res).astype(int).ravel()

        meta_best = build_meta_from_params(best_params_fold)
        meta_best.fit(X_meta_train_res, y_meta_train_res)

        # ==========================
        # 3) Evaluate on the original test_outer set.
        # ==========================
        X_meta_test = make_meta_features(lr_best, et_best, rf_best, X_test_outer)

        y_proba_test = meta_best.predict_proba(X_meta_test)[:, 1]
        thr = best_params_fold["meta_threshold"]
        y_pred_test = (y_proba_test >= thr).astype(int)

        acc  = accuracy_score(y_test_outer, y_pred_test)
        prec = precision_score(y_test_outer, y_pred_test, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred_test, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred_test, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba_test)
        cm   = confusion_matrix(y_test_outer, y_pred_test)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # ==========================
        # 4) Print fold-level results.
        # ==========================
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # ==========================
    # 5) Summary across folds.
    # ==========================
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / tuned params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Example usage:
#
# X = df.drop("target", axis=1)
# y = df["target"]
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
     X, y,
     outer_splits=5,
     inner_splits=3,
     n_trials=100,      # Increase the number of trials for a deeper search.
     seed_outer=42,
     seed_inner=123,
     seed_optuna=42
 )
#
# summary_results: mean/std metrics for reporting.
# fold_metrics: fold-level performance and confusion matrix counts.
# params_each_fold: joint hyperparameters (LR+ET+RF+meta+threshold) for each fold.
# =========================================================


# Tuning hyperparameter tuning using Bayesian optimization RF+ET Novel Stacking

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================
# 1) Utility functions
# =========================

def _to_numpy(X_part):
    # Supports DataFrame or ndarray.
    return X_part.values if hasattr(X_part, "iloc") else X_part

def entropy_binary(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p*np.log(p) + (1-p)*np.log(1-p))

def rf_vote_variance(rf_clf, X_scaled):
    """
    rf_clf: RandomForestClassifier A fitted classifier, not a pipeline.
    X_scaled: Scaled features (ndarray)
    Return uncertainty = p_hat * (1-p_hat)
    where p_hat is the proportion of tree votes.
    """
    votes = np.column_stack([est.predict(X_scaled) for est in rf_clf.estimators_])
    p_hat = votes.mean(axis=1)
    return p_hat * (1.0 - p_hat)


def build_models_from_params(params_joint):
    """
    Create unfitted models from joint parameters proposed by Optuna.
    params_joint  key:
        # ExtraTrees
        et_n_estimators
        et_max_depth
        et_min_samples_split
        et_min_samples_leaf
        et_max_features
        et_bootstrap

        # RandomForest
        rf_n_estimators
        rf_max_depth
        rf_min_samples_split
        rf_min_samples_leaf
        rf_max_features
        rf_bootstrap

        # Meta LR
        meta_C
        meta_max_iter
    """
    et_model = ExtraTreesClassifier(
        n_estimators=params_joint["et_n_estimators"],
        max_depth=params_joint["et_max_depth"],
        min_samples_split=params_joint["et_min_samples_split"],
        min_samples_leaf=params_joint["et_min_samples_leaf"],
        max_features=params_joint["et_max_features"],
        bootstrap=params_joint["et_bootstrap"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf_model = RandomForestClassifier(
        n_estimators=params_joint["rf_n_estimators"],
        max_depth=params_joint["rf_max_depth"],
        min_samples_split=params_joint["rf_min_samples_split"],
        min_samples_leaf=params_joint["rf_min_samples_leaf"],
        max_features=params_joint["rf_max_features"],
        bootstrap=params_joint["rf_bootstrap"],
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    meta_lr = LogisticRegression(
        C=params_joint["meta_C"],
        max_iter=params_joint["meta_max_iter"],
        random_state=RANDOM_STATE
    )

    return et_model, rf_model, meta_lr


# =========================
# 2) 1 training/eval step for inner CV
# =========================
def train_and_eval_once_inner(
    X_tr, y_tr,
    X_val, y_val,
    params_joint
):
    """
    Use only in inner CV.
    - Apply SMOTEENN only to the training data (X_tr, y_tr).
    - Scale separately for ET and RF.
    - Train ET and RF.
    - Create meta-features (probability, entropy/probability, vote variance).
    - Train meta LR.
    - Evaluate recall on the original validation set.
    """
    # 1) Oversample only the training set.
    smoteenn = SMOTEENN(random_state=RANDOM_STATE)
    X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

    X_tr_res_np = _to_numpy(X_tr_res)
    X_val_np    = _to_numpy(X_val)

    # 2) Scale separately for ET and RF.
    scaler_et = StandardScaler()
    scaler_rf = StandardScaler()

    X_tr_res_et = scaler_et.fit_transform(X_tr_res_np)
    X_val_et    = scaler_et.transform(X_val_np)

    X_tr_res_rf = scaler_rf.fit_transform(X_tr_res_np)
    X_val_rf    = scaler_rf.transform(X_val_np)

    # 3) Create models from parameters.
    et_model, rf_model, meta_lr = build_models_from_params(params_joint)

    # 4) Train base learners.
    et_model.fit(X_tr_res_et, y_tr_res)
    rf_model.fit(X_tr_res_rf, y_tr_res)

    # 5) meta-train features
    p_et_tr   = et_model.predict_proba(X_tr_res_et)[:, 1]
    H_et_tr   = entropy_binary(p_et_tr)

    p_rf_tr   = rf_model.predict_proba(X_tr_res_rf)[:, 1]
    Var_rf_tr = rf_vote_variance(rf_model, X_tr_res_rf)

    X_meta_tr = np.column_stack([p_et_tr, H_et_tr, p_rf_tr, Var_rf_tr])
    y_meta_tr = np.asarray(y_tr_res).astype(int).ravel()

    meta_lr.fit(X_meta_tr, y_meta_tr)

    # 6) meta-val features
    p_et_val   = et_model.predict_proba(X_val_et)[:, 1]
    H_et_val   = entropy_binary(p_et_val)

    p_rf_val   = rf_model.predict_proba(X_val_rf)[:, 1]
    Var_rf_val = rf_vote_variance(rf_model, X_val_rf)

    X_meta_val = np.column_stack([p_et_val, H_et_val, p_rf_val, Var_rf_val])

    # 7) predict val
    y_val_proba = meta_lr.predict_proba(X_meta_val)[:, 1]
    y_val_pred  = (y_val_proba >= 0.5).astype(int)

    # 8) Tuning metric = recall.
    return recall_score(y_val, y_val_pred, zero_division=0)


# =========================
# 3) inner-CV evaluator used by Optuna objective
# =========================
def evaluate_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr  = X_train_outer.iloc[inner_tr_idx] if hasattr(X_train_outer,"iloc") else X_train_outer[inner_tr_idx]
        X_val = X_train_outer.iloc[inner_val_idx] if hasattr(X_train_outer,"iloc") else X_train_outer[inner_val_idx]
        y_tr  = y_train_outer.iloc[inner_tr_idx] if hasattr(y_train_outer,"iloc") else y_train_outer[inner_tr_idx]
        y_val = y_train_outer.iloc[inner_val_idx] if hasattr(y_train_outer,"iloc") else y_train_outer[inner_val_idx]

        fold_rec = train_and_eval_once_inner(
            X_tr, y_tr,
            X_val, y_val,
            params_joint
        )
        recalls.append(fold_rec)

    return np.mean(recalls)


# =========================
# 4) Optuna tuning for one outer fold
# =========================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    def objective(trial):
        # ExtraTrees space
        et_n_estimators     = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth        = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split= trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features     = trial.suggest_categorical("et_max_features", ["sqrt", "log2", None])
        et_bootstrap        = trial.suggest_categorical("et_bootstrap", [True, False])

        # RandomForest space
        rf_n_estimators     = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth        = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split= trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features     = trial.suggest_categorical("rf_max_features", ["sqrt", "log2", None])
        rf_bootstrap        = trial.suggest_categorical("rf_bootstrap", [True, False])

        # Meta LR space
        meta_C        = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            "et_n_estimators":      et_n_estimators,
            "et_max_depth":         et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf":  et_min_samples_leaf,
            "et_max_features":      et_max_features,
            "et_bootstrap":         et_bootstrap,

            "rf_n_estimators":      rf_n_estimators,
            "rf_max_depth":         rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf":  rf_min_samples_leaf,
            "rf_max_features":      rf_max_features,
            "rf_bootstrap":         rf_bootstrap,

            "meta_C":        meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_recall = evaluate_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_recall

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint  = study.best_params.copy()
    best_inner_recall  = study.best_value
    return best_params_joint, best_inner_recall


# =========================
# 5) Outer loop (nested CV + report)
# =========================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params  = []

    print("===== Nested CV (Joint Bayesian tuning for ET + RF + Meta LR using uncertainty features) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        # split outer train/test
        X_train_outer = X.iloc[tr_idx] if hasattr(X,"iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X,"iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y,"iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y,"iloc") else y[te_idx]

        # 1) tune joint hyperparams (inner CV on train_outer only)
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx  # Use a different seed for each fold.
        )
        fold_params.append(best_params_fold)

        # 2) retrain using best_params_fold on full train_outer (with SMOTEENN), then test
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        X_train_res_np = _to_numpy(X_train_res)
        X_test_np      = _to_numpy(X_test_outer)

        # Scale separately for ET and RF.
        scaler_et_full = StandardScaler()
        scaler_rf_full = StandardScaler()

        X_train_res_et = scaler_et_full.fit_transform(X_train_res_np)
        X_test_et      = scaler_et_full.transform(X_test_np)

        X_train_res_rf = scaler_rf_full.fit_transform(X_train_res_np)
        X_test_rf      = scaler_rf_full.transform(X_test_np)

        # Create models from the best parameters.
        et_best, rf_best, meta_best = build_models_from_params(best_params_fold)

        # Train base learners.
        et_best.fit(X_train_res_et, y_train_res)
        rf_best.fit(X_train_res_rf, y_train_res)

        # Meta-training features from train_resampled.
        p_et_tr_full   = et_best.predict_proba(X_train_res_et)[:, 1]
        H_et_tr_full   = entropy_binary(p_et_tr_full)

        p_rf_tr_full   = rf_best.predict_proba(X_train_res_rf)[:, 1]
        Var_rf_tr_full = rf_vote_variance(rf_best, X_train_res_rf)

        X_meta_tr_full = np.column_stack([p_et_tr_full, H_et_tr_full,
                                          p_rf_tr_full, Var_rf_tr_full])
        y_meta_tr_full = np.asarray(y_train_res).astype(int).ravel()

        # Train the meta-learner.
        meta_best.fit(X_meta_tr_full, y_meta_tr_full)

        # Meta-test features from the raw test_outer set.
        p_et_te   = et_best.predict_proba(X_test_et)[:, 1]
        H_et_te   = entropy_binary(p_et_te)

        p_rf_te   = rf_best.predict_proba(X_test_rf)[:, 1]
        Var_rf_te = rf_vote_variance(rf_best, X_test_rf)

        X_meta_te = np.column_stack([p_et_te, H_et_te,
                                     p_rf_te, Var_rf_te])

        # Predict.
        y_proba = meta_best.predict_proba(X_meta_te)[:, 1]
        y_pred  = (y_proba >= 0.5).astype(int)

        acc  = accuracy_score(y_test_outer, y_pred)
        prec = precision_score(y_test_outer, y_pred, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba)
        cm   = confusion_matrix(y_test_outer, y_pred)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]), "FP": int(cm[0,1]),
            "FN": int(cm[1,0]), "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # Report fold-level results.
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params for this fold:")
        print(best_params_fold)
        print("-"*100)

    # 3) summary across folds
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        print(f"{m.capitalize():<10} mean={df_res[m].mean():.4f}  std={df_res[m].std(ddof=1):.4f}")

    print("\nPer-fold confusion / best params:")
    for row, params in zip(fold_metrics, fold_params):
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {params}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================
# 6) Usage
# =========================
# Prepare X and y.:
# X = df.drop("target", axis=1)
# y = df["target"]
#
# Then call:
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
         X, y,
         outer_splits=5,
         inner_splits=3,
         n_trials=30,          # Increase to 60 or 100 for a deeper search.
         seed_outer=42,
         seed_inner=123,
         seed_optuna=42        # Base seed; fold_idx is added for each fold.
)
#
# fold_metrics     -> fold-level performance and confusion matrix.
# params_each_fold -> Hyperparameters for the model group (ET+RF+meta) in each fold.


# Tuning New hyperparameter tuning using Bayesian optimization RF+SVM+ET Novel Stacking

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, recall_score
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
# 1) Utilities
# =========================================================
def entropy_binary(p, eps=1e-12):
    """
    Binary probability entropy:
    H(p) = -(p log p + (1-p) log (1-p))
    """
    p = np.clip(p, eps, 1 - eps)
    return -(p*np.log(p) + (1-p)*np.log(1-p))


def rf_vote_variance(rf_pipeline, X):
    """
    Estimate RF uncertainty using the variance of hard votes across trees.
    Steps:
      1) Extract the scaler inside the pipeline and transform X.
      2) Let each tree in the forest predict a hard label (0/1).
      3) Compute vote proportion p_hat = mean(hard votes).
      4) Uncertainty is approximated as p_hat * (1 - p_hat).
    """
    scaler = rf_pipeline.named_steps['standardscaler']
    X_trf = scaler.transform(X.values if hasattr(X, "iloc") else X)
    rf = rf_pipeline.named_steps['randomforestclassifier']
    votes = np.column_stack([est.predict(X_trf) for est in rf.estimators_])
    p_hat = votes.mean(axis=1)
    return p_hat * (1.0 - p_hat)


def build_base_models_from_params(params):
    """
    Build base models (RF, SVM, ET) from params_joint.

    Expected keys in params:
        rf_n_estimators, rf_max_depth, rf_min_samples_split,
        rf_min_samples_leaf, rf_bootstrap, rf_max_features
        svm_C, svm_kernel, svm_gamma
        et_n_estimators, et_max_depth, et_min_samples_split,
        et_min_samples_leaf, et_max_features, et_bootstrap
    """

    rf_model = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=params["rf_n_estimators"],
            max_depth=params["rf_max_depth"],
            min_samples_split=params["rf_min_samples_split"],
            min_samples_leaf=params["rf_min_samples_leaf"],
            bootstrap=params["rf_bootstrap"],
            max_features=params["rf_max_features"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    svm_model = make_pipeline(
        StandardScaler(),
        SVC(
            C=params["svm_C"],
            kernel=params["svm_kernel"],
            gamma=params["svm_gamma"],
            probability=True,
            random_state=RANDOM_STATE
        )
    )

    et_model = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params["et_n_estimators"],
            max_depth=params["et_max_depth"],
            min_samples_split=params["et_min_samples_split"],
            min_samples_leaf=params["et_min_samples_leaf"],
            max_features=params["et_max_features"],
            bootstrap=params["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    return rf_model, svm_model, et_model


def build_meta_from_params(params):
    """
    meta-learner = LogisticRegression
    Expected keys in params:
        meta_C, meta_max_iter
    """
    meta_lr = LogisticRegression(
        C=params["meta_C"],
        max_iter=params["meta_max_iter"],
        random_state=RANDOM_STATE
    )
    return meta_lr


def make_meta_features(rf_model, svm_model, et_model, X_input):
    """
    Create the meta-feature matrix for the meta-learner:
        [ RF_prob, RF_voteVar,
          SVM_prob, SVM_entropy,
          ET_prob, ET_entropy ]
    """

    # RF prob + vote variance
    p_rf = rf_model.predict_proba(X_input)[:, 1]
    Var_rf = rf_vote_variance(rf_model, X_input)

    # SVM prob + entropy
    p_svm = svm_model.predict_proba(X_input)[:, 1]
    H_svm = entropy_binary(p_svm)

    # ET prob + entropy
    p_et = et_model.predict_proba(X_input)[:, 1]
    H_et = entropy_binary(p_et)

    X_meta = np.column_stack([
        p_rf,  Var_rf,
        p_svm, H_svm,
        p_et,  H_et
    ])
    return X_meta


# =========================================================
# 2) Inner CV objective function
#    => Use only train_outer.
#    => Objective: mean recall. (maximize)
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
    Inner CV (e.g., 3-fold) within train_outer.:
      - Split into inner_train and inner_val.
      - Apply SMOTEENN only to inner_train.
      - Train base models according to params_joint.
      - Create meta-features from inner_train_resampled.
      - Train meta LR using params_joint.
      - Evaluate recall on inner_val without oversampling.
      - Store recall for each split and average them.
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample (SMOTEENN) inner-train =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== Train base models using params_joint. =====
        rf_base, svm_base, et_base = build_base_models_from_params(params_joint)
        rf_base.fit(X_tr_res, y_tr_res)
        svm_base.fit(X_tr_res, y_tr_res)
        et_base.fit(X_tr_res, y_tr_res)

        # ===== meta-train features (train_resampled) =====
        X_meta_tr = make_meta_features(rf_base, svm_base, et_base, X_tr_res)
        y_meta_tr = np.asarray(y_tr_res).astype(int).ravel()

        meta_lr = build_meta_from_params(params_joint)
        meta_lr.fit(X_meta_tr, y_meta_tr)

        # ===== Meta-validation features on the raw validation set without oversampling. =====
        X_meta_val = make_meta_features(rf_base, svm_base, et_base, X_val)

        y_val_proba = meta_lr.predict_proba(X_meta_val)[:, 1]
        y_val_pred  = (y_val_proba >= 0.5).astype(int)  # threshold fixed 0.5

        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# 3) Bayesian tuning with Optuna for one outer fold.
#    => Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
    Optuna jointly tunes the following components::
      - RF base
      - SVM base
      - ET base
      - Meta LR
    Threshold is not tuned; use a fixed value of 0.5.

    Objective: maximize mean recall from inner CV.
    """

    def objective(trial):
        # -------- RF params --------
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features = trial.suggest_categorical(
            "rf_max_features", ["sqrt", "log2", None]
        )
        rf_bootstrap = trial.suggest_categorical(
            "rf_bootstrap", [True, False]
        )

        # -------- SVM params --------
        svm_C = trial.suggest_float("svm_C", 1e-4, 1e2, log=True)
        svm_kernel = trial.suggest_categorical("svm_kernel", ["linear", "rbf"])
        svm_gamma = trial.suggest_categorical("svm_gamma", ["scale", "auto"])

        # -------- ET params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            "rf_n_estimators": rf_n_estimators,
            "rf_max_depth": rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf": rf_min_samples_leaf,
            "rf_max_features": rf_max_features,
            "rf_bootstrap": rf_bootstrap,

            "svm_C": svm_C,
            "svm_kernel": svm_kernel,
            "svm_gamma": svm_gamma,

            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_rec = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_rec

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()
    best_inner_recall = study.best_value  # Mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# 4) MAIN: Outer nested CV and fold-level reporting.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits: Number of folds for final evaluation (e.g., 5).
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials: Number of Optuna trials per outer fold.
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV (joint Bayesian tuning for RF+SVM+ET base -> LR meta, uncertainty features) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        # --------------------------
        # Split data for this outer fold.
        # --------------------------
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # --------------------------
        # 1) Tune only on train_outer using inner CV.
        # --------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx  # Use a different seed for each fold.
        )

        fold_params.append(best_params_fold)

        # --------------------------
        # 2) Retrain final models using tuned parameters on the full train_outer set.
        #    Oversample train_outer before final training.
        # --------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        rf_best, svm_best, et_best = build_base_models_from_params(best_params_fold)
        rf_best.fit(X_train_res, y_train_res)
        svm_best.fit(X_train_res, y_train_res)
        et_best.fit(X_train_res, y_train_res)

        # meta-train
        X_meta_train_res = make_meta_features(rf_best, svm_best, et_best, X_train_res)
        y_meta_train_res = np.asarray(y_train_res).astype(int).ravel()

        meta_best = build_meta_from_params(best_params_fold)
        meta_best.fit(X_meta_train_res, y_meta_train_res)

        # --------------------------
        # 3) Evaluate on the raw test_outer set without oversampling.
        # --------------------------
        X_meta_test = make_meta_features(rf_best, svm_best, et_best, X_test_outer)

        y_proba_test = meta_best.predict_proba(X_meta_test)[:, 1]
        y_pred_test  = (y_proba_test >= 0.5).astype(int)  # threshold fix 0.5

        acc  = accuracy_score(y_test_outer, y_pred_test)
        prec = precision_score(y_test_outer, y_pred_test, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred_test, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred_test, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba_test)
        cm   = confusion_matrix(y_test_outer, y_pred_test)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # --------------------------
        # 4) Display fold-level results.
        # --------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # --------------------------
    # 5) Summarize results across all outer folds.
    # --------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / tuned params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Example usage:
#
# X = df.drop("target", axis=1)
# y = df["target"]
#
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
     X, y,
     outer_splits=5,
     inner_splits=3,
     n_trials=100,      # Increase trials for deeper tuning.
     seed_outer=42,
     seed_inner=123,
     seed_optuna=42
)
#
# summary_results   -> mean/std metrics across folds.
# fold_metrics: fold-level performance and confusion matrix counts. matrix
# params_each_fold  -> Tuned hyperparameters for each fold.
# =========================================================


# Tuning hyperparameter tuning using Bayesian optimization RF+LR+SVM+ET Novel stacking

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, recall_score
)
from imblearn.combine import SMOTEENN

import optuna
from optuna.samplers import TPESampler


# =========================================================
# 1) Utilities
# =========================================================
def entropy_binary(p, eps=1e-12):
    """
    Binary probability entropy.:
    H(p) = -(p log p + (1-p) log (1-p))
    """
    p = np.clip(p, eps, 1 - eps)
    return -(p*np.log(p) + (1-p)*np.log(1-p))


def rf_vote_variance(rf_pipeline, X):
    """
    Approximate RF uncertainty using vote variance across trees.
    Steps:
      1) Extract the scaler from the pipeline and transform X.
      2) Let each tree in the forest predict a hard label (0/1).
      3) Compute p_hat = mean(hard votes).
      4) Return p_hat * (1 - p_hat).
    """
    scaler = rf_pipeline.named_steps['standardscaler']
    X_trf = scaler.transform(X.values if hasattr(X, "iloc") else X)
    rf = rf_pipeline.named_steps['randomforestclassifier']
    votes = np.column_stack([est.predict(X_trf) for est in rf.estimators_])
    p_hat = votes.mean(axis=1)
    return p_hat * (1.0 - p_hat)


def build_base_models_from_params(params):
    """
    Build base models (RF, LR, SVM, ET) from params_joint.

    Required keys in params:
      RF:
        rf_n_estimators, rf_max_depth, rf_min_samples_split,
        rf_min_samples_leaf, rf_bootstrap, rf_max_features
      LR:
        lr_C, lr_max_iter
      SVM:
        svm_C, svm_kernel, svm_gamma
      ET:
        et_n_estimators, et_max_depth, et_min_samples_split,
        et_min_samples_leaf, et_max_features, et_bootstrap
    """

    rf_model = make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=params["rf_n_estimators"],
            max_depth=params["rf_max_depth"],
            min_samples_split=params["rf_min_samples_split"],
            min_samples_leaf=params["rf_min_samples_leaf"],
            bootstrap=params["rf_bootstrap"],
            max_features=params["rf_max_features"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    lr_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=params["lr_C"],
            max_iter=params["lr_max_iter"],
            random_state=RANDOM_STATE
        )
    )

    svm_model = make_pipeline(
        StandardScaler(),
        SVC(
            C=params["svm_C"],
            kernel=params["svm_kernel"],
            gamma=params["svm_gamma"],
            probability=True,
            random_state=RANDOM_STATE
        )
    )

    et_model = make_pipeline(
        StandardScaler(),
        ExtraTreesClassifier(
            n_estimators=params["et_n_estimators"],
            max_depth=params["et_max_depth"],
            min_samples_split=params["et_min_samples_split"],
            min_samples_leaf=params["et_min_samples_leaf"],
            max_features=params["et_max_features"],
            bootstrap=params["et_bootstrap"],
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )

    return rf_model, lr_model, svm_model, et_model


def build_meta_from_params(params):
    """
    meta-learner = LogisticRegression
    Required parameters:
      meta_C, meta_max_iter
    """
    meta_lr = LogisticRegression(
        C=params["meta_C"],
        max_iter=params["meta_max_iter"],
        random_state=RANDOM_STATE
    )
    return meta_lr


def make_meta_features(rf_model, lr_model, svm_model, et_model, X_input):
    """
    Create the meta-feature matrix for meta LR.:
      [ RF_prob, RF_voteVar,
        LR_prob, LR_entropy,
        SVM_prob, SVM_entropy,
        ET_prob, ET_entropy ]
    """

    # RF: probability and vote variance across trees.
    p_rf = rf_model.predict_proba(X_input)[:, 1]
    Var_rf = rf_vote_variance(rf_model, X_input)

    # LR: prob + entropy
    p_lr = lr_model.predict_proba(X_input)[:, 1]
    H_lr = entropy_binary(p_lr)

    # SVM: prob + entropy
    p_svm = svm_model.predict_proba(X_input)[:, 1]
    H_svm = entropy_binary(p_svm)

    # ET: prob + entropy
    p_et = et_model.predict_proba(X_input)[:, 1]
    H_et = entropy_binary(p_et)

    X_meta = np.column_stack([
        p_rf,  Var_rf,
        p_lr,  H_lr,
        p_svm, H_svm,
        p_et,  H_et
    ])
    return X_meta


# =========================================================
# 2) Inner CV objective:
#    - Use only train_outer.
#    - Objective: mean recall. (maximize)
# =========================================================
def evaluate_joint_params_inner_cv(
    X_train_outer,
    y_train_outer,
    params_joint,
    n_splits_inner=3,
    random_state_inner=123
):
    """
    Perform inner CV (e.g., 3-fold) within train_outer.:
      - Split into inner_train and inner_val.
      - SMOTEENN  inner_train
      - Train base models from params_joint.
      - Create meta-features from inner_train_resampled.
      -  meta LR  params_joint
      - Predict on raw inner_val without oversampling.
      - Compute recall and average it.
    """
    kf_inner = StratifiedKFold(
        n_splits=n_splits_inner,
        shuffle=True,
        random_state=random_state_inner
    )

    recalls = []

    for inner_tr_idx, inner_val_idx in kf_inner.split(X_train_outer, y_train_outer):
        X_tr, X_val = (
            X_train_outer.iloc[inner_tr_idx],
            X_train_outer.iloc[inner_val_idx],
        )
        y_tr, y_val = (
            y_train_outer.iloc[inner_tr_idx],
            y_train_outer.iloc[inner_val_idx],
        )

        # ===== Oversample only the inner-training set. =====
        smoteenn = SMOTEENN(random_state=42)
        X_tr_res, y_tr_res = smoteenn.fit_resample(X_tr, y_tr)

        # ===== Train base models using params_joint. =====
        rf_base, lr_base, svm_base, et_base = build_base_models_from_params(params_joint)
        rf_base.fit(X_tr_res, y_tr_res)
        lr_base.fit(X_tr_res, y_tr_res)
        svm_base.fit(X_tr_res, y_tr_res)
        et_base.fit(X_tr_res, y_tr_res)

        # ===== Meta-training features on train_resampled. =====
        X_meta_tr = make_meta_features(rf_base, lr_base, svm_base, et_base, X_tr_res)
        y_meta_tr = np.asarray(y_tr_res).astype(int).ravel()

        meta_lr = build_meta_from_params(params_joint)
        meta_lr.fit(X_meta_tr, y_meta_tr)

        # ===== Meta-validation features on raw validation data without oversampling. =====
        X_meta_val = make_meta_features(rf_base, lr_base, svm_base, et_base, X_val)

        y_val_proba = meta_lr.predict_proba(X_meta_val)[:, 1]
        y_val_pred  = (y_val_proba >= 0.5).astype(int)  # threshold fixed 0.5

        rec = recall_score(y_val, y_val_pred, zero_division=0)
        recalls.append(rec)

    return np.mean(recalls)


# =========================================================
# 3) Bayesian tuning with Optuna for one outer fold.
#    => Return best_params_joint and best_inner_recall.
# =========================================================
def tune_joint_params_for_outer_fold(
    X_train_outer,
    y_train_outer,
    n_trials=30,
    n_splits_inner=3,
    random_state_inner=123,
    seed_optuna=42
):
    """
    Jointly tune all components::
      - RF base
      - LR base
      - SVM base
      - ET base
      - Meta LR
    Threshold is not tuned; use a fixed value of 0.5.
    objective = maximize mean recall  inner CV
    """

    def objective(trial):
        # -------- RF params --------
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300)
        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 50)
        rf_min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 20)
        rf_min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 10)
        rf_max_features = trial.suggest_categorical(
            "rf_max_features", ["sqrt", "log2", None]
        )
        rf_bootstrap = trial.suggest_categorical(
            "rf_bootstrap", [True, False]
        )

        # -------- LR params --------
        lr_C = trial.suggest_float("lr_C", 1e-4, 1e2, log=True)
        lr_max_iter = trial.suggest_int("lr_max_iter", 100, 2000)

        # -------- SVM params --------
        svm_C = trial.suggest_float("svm_C", 1e-4, 1e2, log=True)
        svm_kernel = trial.suggest_categorical("svm_kernel", ["linear", "rbf"])
        svm_gamma = trial.suggest_categorical("svm_gamma", ["scale", "auto"])

        # -------- ET params --------
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300)
        et_max_depth = trial.suggest_int("et_max_depth", 3, 50)
        et_min_samples_split = trial.suggest_int("et_min_samples_split", 2, 20)
        et_min_samples_leaf = trial.suggest_int("et_min_samples_leaf", 1, 10)
        et_max_features = trial.suggest_categorical(
            "et_max_features", ["sqrt", "log2", None]
        )
        et_bootstrap = trial.suggest_categorical(
            "et_bootstrap", [True, False]
        )

        # -------- Meta LR params --------
        meta_C = trial.suggest_float("meta_C", 1e-4, 1e2, log=True)
        meta_max_iter = trial.suggest_int("meta_max_iter", 100, 2000)

        params_joint = {
            "rf_n_estimators": rf_n_estimators,
            "rf_max_depth": rf_max_depth,
            "rf_min_samples_split": rf_min_samples_split,
            "rf_min_samples_leaf": rf_min_samples_leaf,
            "rf_max_features": rf_max_features,
            "rf_bootstrap": rf_bootstrap,

            "lr_C": lr_C,
            "lr_max_iter": lr_max_iter,

            "svm_C": svm_C,
            "svm_kernel": svm_kernel,
            "svm_gamma": svm_gamma,

            "et_n_estimators": et_n_estimators,
            "et_max_depth": et_max_depth,
            "et_min_samples_split": et_min_samples_split,
            "et_min_samples_leaf": et_min_samples_leaf,
            "et_max_features": et_max_features,
            "et_bootstrap": et_bootstrap,

            "meta_C": meta_C,
            "meta_max_iter": meta_max_iter
        }

        mean_rec = evaluate_joint_params_inner_cv(
            X_train_outer,
            y_train_outer,
            params_joint,
            n_splits_inner=n_splits_inner,
            random_state_inner=random_state_inner
        )

        # maximize recall
        return mean_rec

    # run Optuna
    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed_optuna),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    best_params_joint = study.best_params.copy()
    best_inner_recall = study.best_value  # Best mean recall from inner CV.

    return best_params_joint, best_inner_recall


# =========================================================
# 4) MAIN: Outer nested CV and fold-level summary.
# =========================================================
def nested_cv_pipeline_stacking_joint(
    X,
    y,
    outer_splits=5,
    inner_splits=3,
    n_trials=30,
    seed_outer=42,
    seed_inner=123,
    seed_optuna=42
):
    """
    outer_splits:    fold  ( 5)
    inner_splits: Number of folds for tuning (e.g., 3).
    n_trials: Number of Optuna trials per outer fold.
    """

    outer_kf = StratifiedKFold(
        n_splits=outer_splits,
        shuffle=True,
        random_state=seed_outer
    )

    fold_metrics = []
    fold_params = []

    print("===== Nested CV (joint Bayesian tuning for RF+LR+SVM+ET base -> LR meta, with uncertainty features) =====\n")

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_kf.split(X, y), start=1):
        # --------------------------
        # Split data for this outer fold.
        # --------------------------
        X_train_outer = X.iloc[tr_idx] if hasattr(X, "iloc") else X[tr_idx]
        X_test_outer  = X.iloc[te_idx] if hasattr(X, "iloc") else X[te_idx]
        y_train_outer = y.iloc[tr_idx] if hasattr(y, "iloc") else y[tr_idx]
        y_test_outer  = y.iloc[te_idx] if hasattr(y, "iloc") else y[te_idx]

        # --------------------------
        # 1) Tune with Optuna only on train_outer.
        # --------------------------
        best_params_fold, best_inner_recall = tune_joint_params_for_outer_fold(
            X_train_outer,
            y_train_outer,
            n_trials=n_trials,
            n_splits_inner=inner_splits,
            random_state_inner=seed_inner,
            seed_optuna=seed_optuna + fold_idx  # Use a different seed for each fold.
        )

        fold_params.append(best_params_fold)

        # --------------------------
        # 2) Train final models using best_params_fold.
        #    on resampled train_outer and evaluate on test_outer.
        # --------------------------
        smoteenn = SMOTEENN(random_state=RANDOM_STATE)
        X_train_res, y_train_res = smoteenn.fit_resample(X_train_outer, y_train_outer)

        rf_best, lr_best, svm_best, et_best = build_base_models_from_params(best_params_fold)
        rf_best.fit(X_train_res, y_train_res)
        lr_best.fit(X_train_res, y_train_res)
        svm_best.fit(X_train_res, y_train_res)
        et_best.fit(X_train_res, y_train_res)

        # Meta-training on resampled data.
        X_meta_train_res = make_meta_features(rf_best, lr_best, svm_best, et_best, X_train_res)
        y_meta_train_res = np.asarray(y_train_res).astype(int).ravel()

        meta_best = build_meta_from_params(best_params_fold)
        meta_best.fit(X_meta_train_res, y_meta_train_res)

        # --------------------------
        # 3) Evaluate on the original test_outer set without oversampling.
        # --------------------------
        X_meta_test = make_meta_features(rf_best, lr_best, svm_best, et_best, X_test_outer)

        y_proba_test = meta_best.predict_proba(X_meta_test)[:, 1]
        y_pred_test  = (y_proba_test >= 0.5).astype(int)  # threshold fixed 0.5

        acc  = accuracy_score(y_test_outer, y_pred_test)
        prec = precision_score(y_test_outer, y_pred_test, zero_division=0)
        rec  = recall_score(y_test_outer, y_pred_test, zero_division=0)
        f1   = f1_score(y_test_outer, y_pred_test, zero_division=0)
        auc  = roc_auc_score(y_test_outer, y_proba_test)
        cm   = confusion_matrix(y_test_outer, y_pred_test)

        fold_metrics.append({
            "fold": fold_idx,
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "auc": auc,
            "TN": int(cm[0,0]),
            "FP": int(cm[0,1]),
            "FN": int(cm[1,0]),
            "TP": int(cm[1,1]),
            "best_inner_recall": best_inner_recall
        })

        # --------------------------
        # 4) Display fold-level results.
        # --------------------------
        print(f"[Outer Fold {fold_idx}]")
        print(f"  Inner-CV tuned recall (objective): {best_inner_recall:.4f}")
        print(f"  Test Performance on this fold:")
        print(f"    Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  "
              f"F1={f1:.4f}  AUC={auc:.4f}")
        print("  Confusion Matrix:")
        print(cm)
        print("  Best Joint Params:")
        print(best_params_fold)
        print("-"*100)

    # --------------------------
    # 5) Summarize results across all outer folds.
    # --------------------------
    df_res = pd.DataFrame(fold_metrics)

    print("\n===== Summary over all outer folds =====")
    for m in ["accuracy", "precision", "recall", "f1", "auc"]:
        mean_val = df_res[m].mean()
        std_val  = df_res[m].std(ddof=1)
        print(f"{m.capitalize():<10} mean={mean_val:.4f}  std={std_val:.4f}")

    print("\nPer-fold confusion / tuned params:")
    for row in fold_metrics:
        print(f" Fold {row['fold']}: "
              f"TN={row['TN']} FP={row['FP']} FN={row['FN']} TP={row['TP']}")
        print(f"  Params: {fold_params[row['fold']-1]}")
        print()

    summary_results = {
        "Accuracy":  (df_res["accuracy"].mean(),  df_res["accuracy"].std(ddof=1)),
        "Precision": (df_res["precision"].mean(), df_res["precision"].std(ddof=1)),
        "Recall":    (df_res["recall"].mean(),    df_res["recall"].std(ddof=1)),
        "F1":        (df_res["f1"].mean(),        df_res["f1"].std(ddof=1)),
        "AUC":       (df_res["auc"].mean(),       df_res["auc"].std(ddof=1)),
    }

    return summary_results, fold_metrics, fold_params


# =========================================================
# Example usage:
#
# X = df.drop("target", axis=1)
# y = df["target"]
#
summary_results, fold_metrics, params_each_fold = nested_cv_pipeline_stacking_joint(
     X, y,
     outer_splits=5,
     inner_splits=3,
     n_trials=100,      # Number of Optuna trials per fold; increase for deeper tuning.
     seed_outer=42,
     seed_inner=123,
     seed_optuna=42
 )
#
# summary_results   -> mean/std metrics across folds.
# fold_metrics: fold-level performance and confusion matrix counts. matrix
# params_each_fold  -> Tuned hyperparameters for each fold.
# =========================================================
